# Stop guessing: add data-value context

Schema context tells a model that `Activity Type Code` is a string column, but not which strings are stored in it. This notebook collects a small, reviewed set of categorical values so the prompt can use exact labels without sending table rows or high-cardinality columns.

It uses the persistent DuckDB database from `03a_duckdb_settings.ipynb`. The local checkpoint does not make a model request.

In [ ]:
import os
import re
from dataclasses import dataclass
from pathlib import Path

import ibis
import pandas as pd
from openai import OpenAI

## Connect to the prepared table

We reuse the DuckDB file and table built in the data-preparation tutorial. Keeping the connection here makes this notebook runnable on its own, while T05 remains the source for the fuller schema-and-dialect discussion.

In [ ]:
def connect_to_prepared_database():
    database_directory = os.getenv("DUCKDB_DATABASES_PATH")
    if not database_directory:
        raise EnvironmentError(
            "Set DUCKDB_DATABASES_PATH to the directory used in the data tutorial."
        )

    database_path = Path(database_directory).expanduser() / "sql-ai-agent.duckdb"
    if not database_path.is_file():
        raise FileNotFoundError(
            f"Could not find {database_path}. Run 03a_duckdb_settings.ipynb first."
        )

    try:
        connection = ibis.duckdb.connect(database_path, read_only=True)
    except Exception as error:
        if "Could not set lock" in str(error):
            raise RuntimeError(
                "The DuckDB file is already open for writing. Shut down the kernel "
                "that opened it, then restart this notebook kernel and try again."
            ) from error
        raise
    table_name = "air_traffic"
    if table_name not in connection.list_tables():
        raise RuntimeError(f"Could not find the {table_name!r} table.")

    return connection, table_name, database_path


connection, table_name, database_path = connect_to_prepared_database()
print(f"Connected to {database_path.name}; available table: {table_name}")

## Define the value-context policy before collecting values

The policy is an allowlist, not automatic sensitive-data detection. Before adding a column, we must review whether its values are relevant and permitted to send to a model. The per-column and total-value limits make the prompt input predictable; they are a value budget, not an exact tokenizer count.

In [ ]:
@dataclass(frozen=True)
class ValueContextPolicy:
    allowed_columns: tuple[str, ...]
    max_values_per_column: int
    max_total_values: int


value_policy = ValueContextPolicy(
    allowed_columns=(
        "GEO Summary",
        "GEO Region",
        "Activity Type Code",
        "Price Category Code",
        "Terminal",
        "Boarding Area",
        "Operating Airline",
    ),
    max_values_per_column=10,
    max_total_values=32,
)

requested_columns = (
    "GEO Summary",
    "GEO Region",
    "Activity Type Code",
    "Price Category Code",
    "Terminal",
    "Boarding Area",
    "Operating Airline",
)

## Collect bounded distinct values

This helper checks the policy before building an identifier into SQL, counts values before retrieving them, and keeps a local record of skipped columns. Only accepted values become prompt context.

In [ ]:
def quote_identifier(identifier):
    return '"' + identifier.replace('"', '""') + '"'


def collect_value_context(connection, table_name, columns, policy):
    if not table_name.isidentifier():
        raise ValueError("This tutorial accepts a simple table name only.")

    approved_columns = set(policy.allowed_columns)
    unknown_columns = set(columns) - approved_columns
    if unknown_columns:
        unknown = ", ".join(sorted(unknown_columns))
        raise ValueError(f"Columns are not approved for prompt context: {unknown}")

    values_by_column = {}
    skipped_columns = {}
    selected_value_count = 0

    for column in columns:
        quoted_column = quote_identifier(column)
        count_sql = f"""
        SELECT COUNT(DISTINCT {quoted_column}) AS value_count
        FROM {quote_identifier(table_name)}
        WHERE {quoted_column} IS NOT NULL
        """
        distinct_count = int(connection.sql(count_sql).execute().iloc[0, 0])

        exceeds_column_limit = distinct_count > policy.max_values_per_column
        exceeds_total_limit = (
            selected_value_count + distinct_count > policy.max_total_values
        )
        if exceeds_column_limit or exceeds_total_limit:
            skipped_columns[column] = distinct_count
            continue

        values_sql = f"""
        SELECT DISTINCT {quoted_column} AS value
        FROM {quote_identifier(table_name)}
        WHERE {quoted_column} IS NOT NULL
        ORDER BY value
        """
        values = connection.sql(values_sql).execute()["value"].tolist()
        values_by_column[column] = tuple(values)
        selected_value_count += len(values)

    return values_by_column, skipped_columns


def format_value_context(values_by_column):
    lines = ["Known stored categorical values (not table rows):"]
    for column, values in values_by_column.items():
        formatted_values = ", ".join(repr(value) for value in values)
        lines.append(f"- {quote_identifier(column)}: {formatted_values}")
    return "\n".join(lines)


values_by_column, skipped_columns = collect_value_context(
    connection, table_name, requested_columns, value_policy
)
value_context = format_value_context(values_by_column)
print(value_context)
print(f"Skipped outside the prompt: {skipped_columns}")
assert sum(len(values) for values in values_by_column.values()) <= 32
assert all(len(values) <= 10 for values in values_by_column.values())
assert skipped_columns == {"Operating Airline": 140}
assert values_by_column["Activity Type Code"] == (
    "Deplaned",
    "Enplaned",
    "Thru / Transit",
)

## Verify the allowlist boundary

A caller cannot add an arbitrary column to the prompt just by passing its name. This example rejects `Passenger Count` because it is not part of the reviewed categorical-value policy.

In [ ]:
try:
    collect_value_context(connection, table_name, ("Passenger Count",), value_policy)
except ValueError as error:
    print(type(error).__name__, error)
else:
    raise AssertionError("An unapproved column unexpectedly entered the prompt context.")

## Use the exact stored value

A question about transit passengers could lead a model to guess the label `Transit`. The bounded context shows that DuckDB stores `Thru / Transit` instead. The guessed query is valid SQL but has no matching rows; the exact value returns a result.

In [ ]:
guessed_value_sql = """
SELECT SUM("Passenger Count") AS "total_passengers"
FROM "air_traffic"
WHERE "Year" = 2024
  AND "Activity Type Code" = 'Transit'
"""

exact_value_sql = """
SELECT SUM("Passenger Count") AS "total_passengers"
FROM "air_traffic"
WHERE "Year" = 2024
  AND "Activity Type Code" = 'Thru / Transit'
"""

guessed_value_result = connection.sql(guessed_value_sql).execute()
exact_value_result = connection.sql(exact_value_sql).execute()
print("Guessed value:")
print(guessed_value_result.to_string(index=False))
print("\nExact stored value:")
print(exact_value_result.to_string(index=False))
assert pd.isna(guessed_value_result.iloc[0, 0])
assert int(exact_value_result.iloc[0, 0]) == 77_159

## Extend the schema-aware agent function

T05 establishes `sql_ai_agent()` as the complete question → prompt → SQL → result loop. This tutorial keeps the same function shape and passes the bounded value block as ordered context after the schema and dialect blocks. The test client exercises that complete path without an API request.

In [ ]:
def build_schema_context(connection, table_name):
    if not table_name.isidentifier():
        raise ValueError("This tutorial accepts a simple table name only.")

    schema_sql = f"""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = '{table_name}'
    ORDER BY ordinal_position
    """
    schema_details = connection.sql(schema_sql).execute()
    if schema_details.empty:
        raise RuntimeError(f"No schema metadata found for {table_name!r}.")

    columns = "\n".join(
        f"    {quote_identifier(row.column_name)} {row.data_type}"
        for row in schema_details.itertuples(index=False)
    )
    return f"CREATE TABLE {quote_identifier(table_name)} (\n{columns}\n);"


DIALECT_GUIDANCE = {
    "DuckDB": (
        "Use DuckDB SQL. Double-quote table and column identifiers, "
        "use LIMIT to restrict rows, and do not use SQL Server TOP syntax."
    )
}


def build_dialect_context(dialect):
    try:
        guidance = DIALECT_GUIDANCE[dialect]
    except KeyError as error:
        supported = ", ".join(DIALECT_GUIDANCE)
        raise ValueError(f"Unsupported dialect {dialect!r}. Choose one of: {supported}.") from error

    return f"Database dialect: {dialect}. {guidance}"


def build_system_prompt(schema_context, dialect_context, context_blocks=()):
    if any(not isinstance(block, str) or not block.strip() for block in context_blocks):
        raise ValueError("Each context block must be a non-empty string.")

    prompt_sections = [
        "\n".join(
            [
                "Translate the user's question into one read-only SQL query.",
                "Return only SQL as plain text, without Markdown or an explanation.",
                "Use only the table and columns supplied below.",
            ]
        ),
        dialect_context,
        f"Available schema:\n{schema_context}",
    ]
    prompt_sections.extend(context_blocks)
    return "\n\n".join(prompt_sections)


def extract_sql(text):
    fenced_sql = re.search(
        r"```(?:sql)?\s*(.*?)```",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )
    sql = fenced_sql.group(1) if fenced_sql else text
    sql = sql.strip()
    if not sql:
        raise ValueError("The model did not return a SQL query.")
    return sql


def sql_ai_agent(
    question,
    client,
    connection,
    table_name,
    model="gpt-5.4-mini",
    dialect="DuckDB",
    context_blocks=(),
):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("question must be a non-empty string.")

    schema_context = build_schema_context(connection, table_name)
    dialect_context = build_dialect_context(dialect)
    system_prompt = build_system_prompt(
        schema_context,
        dialect_context,
        context_blocks=context_blocks,
    )
    response = client.responses.create(
        model=model,
        instructions=system_prompt,
        input=f"Write a SQL query that answers: {question}",
    )
    sql = extract_sql(response.output_text)
    result = connection.sql(sql).execute()

    separator = "-" * 72
    print(f"Question:\n{question}\n")
    print(f"Generated SQL:\n{separator}\n{sql}\n{separator}")
    print("\nResult:")
    print(result.to_string(index=False))

    return sql, result


class FakeResponse:
    def __init__(self, output_text):
        self.output_text = output_text


class FakeResponses:
    def __init__(self, output_text):
        self.output_text = output_text
        self.request = None

    def create(self, **kwargs):
        self.request = kwargs
        return FakeResponse(self.output_text)


class FakeClient:
    def __init__(self, output_text):
        self.responses = FakeResponses(output_text)


data_value_context = (
    "Use exact stored values only when they are listed in the context.\n\n"
    f"{value_context}"
)
schema_context = build_schema_context(connection, table_name)
dialect_context = build_dialect_context("DuckDB")
system_prompt = build_system_prompt(
    schema_context,
    dialect_context,
    context_blocks=(data_value_context,),
)
assert "Thru / Transit" in system_prompt
assert system_prompt.index("Available schema:") < system_prompt.index("Known stored categorical values")

fake_client = FakeClient(f"```sql\n{exact_value_sql.strip()}\n```")
generated_sql, agent_result = sql_ai_agent(
    "What was the total passenger count for transit passengers in 2024?",
    client=fake_client,
    connection=connection,
    table_name=table_name,
    model="test-model",
    context_blocks=(data_value_context,),
)
assert generated_sql == exact_value_sql.strip()
assert int(agent_result.iloc[0, 0]) == 77_159
assert "DuckDB" in fake_client.responses.request["instructions"]
assert "Thru / Transit" in fake_client.responses.request["instructions"]

if os.getenv("OPENAI_API_KEY"):
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    print("OpenAI client is ready. Call sql_ai_agent() with the live client to make a request.")
else:
    print("Set OPENAI_API_KEY to enable a live model request.")

## Checkpoint

We now add only selected stored values to the agent context and can explain why each value is present or omitted. The next tutorial adds business definitions and rules that the database cannot provide.